# Benchmark: LogFolder Flush Latency vs Row Count

Tests how long `flush()` takes for 100 / 1k / 10k / 100k / 1M / 10M rows, each with 10 float64 columns.

Findings help verify whether the adaptive `save_delay_secs` thresholds in `_DataHandler._run()` are safe (i.e. write time < debounce window).


In [2]:
import sys
import time
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from logqbit.logfolder import LogFolder

COLS = [f"c{i}" for i in range(10)]  # 10 float64 columns
ROW_COUNTS = [100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000]
BYTES_PER_ROW = len(COLS) * 8  # float64 = 8 bytes

def expected_delay(n: int) -> float:
    if n < 1_000:   return 0.1
    if n < 10_000:  return 0.2
    if n < 100_000: return 0.5
    return 1.0

def fmt_bytes(b: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if b < 1024:
            return f"{b:.1f} {unit}"
        b /= 1024
    return f"{b:.1f} PB"

results = []

print(f"{'Rows':>10}  {'Write (s)':>10}  {'Delay (s)':>10}  {'Safe?':>6}  "
      f"{'File size':>10}  {'Disk write':>12}")
print("-" * 70)

with tempfile.TemporaryDirectory() as tmp_root:
    for n in ROW_COUNTS:
        folder = Path(tmp_root) / f"bench_{n}"
        folder.mkdir()
        lf = LogFolder(folder / "0", title="bench", create=True)

        # Bulk-load data — testing write/flush latency only, not add_row overhead
        rng = np.random.default_rng(42)
        chunk = 100_000
        rows_left = n
        while rows_left > 0:
            batch = min(chunk, rows_left)
            lf._handler.add_multi_rows(pd.DataFrame({c: rng.random(batch) for c in COLS}))
            rows_left -= batch

        t0 = time.perf_counter()
        lf._handler.flush(timeout=120.0)
        elapsed = time.perf_counter() - t0

        file_bytes = lf.df_path.stat().st_size if lf.df_path.exists() else 0
        delay = expected_delay(n)
        safe = "✓" if elapsed < delay else "✗ !"

        print(f"{n:>10,}  {elapsed:>10.3f}  {delay:>10.1f}  {safe:>6}  "
              f"{fmt_bytes(file_bytes):>10}  {fmt_bytes(n * BYTES_PER_ROW):>12}")

        results.append(dict(n=n, elapsed=elapsed, delay=delay, file_bytes=file_bytes))

        lf._handler.stop()
        lf._handler._thread.join(timeout=5)

      Rows   Write (s)   Delay (s)   Safe?   File size    Disk write
----------------------------------------------------------------------
       100       0.007         0.1       ✓     12.5 KB        7.8 KB
     1,000       0.002         0.2       ✓     82.8 KB       78.1 KB
    10,000       0.004         0.5       ✓    786.0 KB      781.2 KB
   100,000       0.007         1.0       ✓      7.6 MB        7.6 MB
 1,000,000       0.070         1.0       ✓     80.1 MB       76.3 MB
10,000,000       0.854         1.0       ✓    801.3 MB      762.9 MB
